# Day 1 — Reading Refrigerant Properties (CoolProp)
**Mahesha Gonal — Refrigeration Simulation Portfolio**

**In plain terms:** before you can simulate a refrigeration cycle, you need accurate answers to two basic questions — *"at what temperature does this refrigerant boil at a given pressure?"* and *"how much heat does it absorb while boiling?"*

This notebook uses **CoolProp** — a free, open-source thermodynamic properties library (the same kind of database REFPROP provides, but free and usable inside Python) — to pull these numbers directly for **R134a**, and compares them against **R600a (isobutane)**, the refrigerant used in IFB/LG household refrigerators.

No simulation logic yet — this is the foundation every later day builds on.

In [1]:
# CELL 1 — Install dependencies (run this first, every time)
!pip install coolprop matplotlib numpy PyGithub -q

In [2]:
# CELL 2 — Imports
import CoolProp.CoolProp as CP

In [3]:
# CELL 3 — R134a at 1 bar evaporator pressure
P_evap = 1e5  # 1 bar in Pascals

T_sat = CP.PropsSI('T', 'P', P_evap, 'Q', 0, 'R134a')
print(f"R134a saturation temp at 1 bar:   {T_sat - 273.15:.2f} C")

h_liq = CP.PropsSI('H', 'P', P_evap, 'Q', 0, 'R134a')
print(f"Enthalpy saturated liquid (h4):   {h_liq/1000:.2f} kJ/kg")

h_vap = CP.PropsSI('H', 'P', P_evap, 'Q', 1, 'R134a')
print(f"Enthalpy saturated vapour (h1):   {h_vap/1000:.2f} kJ/kg")

latent = (h_vap - h_liq) / 1000
print(f"Latent heat of vaporisation:      {latent:.2f} kJ/kg")

R134a saturation temp at 1 bar:   -26.36 C
Enthalpy saturated liquid (h4):   165.44 kJ/kg
Enthalpy saturated vapour (h1):   382.60 kJ/kg
Latent heat of vaporisation:      217.16 kJ/kg


In [4]:
# CELL 4 — R134a vs R600a comparison
refrigerants = ['R134a', 'R600a']
print(f"\n{'Refrigerant':<12} {'T_sat (C)':<12} {'Latent (kJ/kg)'}")
print("-" * 38)
for ref in refrigerants:
    T = CP.PropsSI('T', 'P', 1e5, 'Q', 0, ref) - 273.15
    h_l = CP.PropsSI('H', 'P', 1e5, 'Q', 0, ref) / 1000
    h_v = CP.PropsSI('H', 'P', 1e5, 'Q', 1, ref) / 1000
    print(f"{ref:<12} {T:<12.2f} {h_v - h_l:.2f}")


Refrigerant  T_sat (C)    Latent (kJ/kg)
--------------------------------------
R134a        -26.36       217.16
R600a        -12.08       365.40


**What these numbers mean:**

- **Saturation temperature** is the boiling point of the refrigerant at that pressure — this is the temperature the evaporator coil sits at, which is what actually cools your cabin air.
- **Latent heat** is how much heat 1 kg of refrigerant soaks up while it boils (evaporates) inside the evaporator. This is the "useful cooling" part of the cycle.

R600a's latent heat is noticeably higher than R134a's. That means **less refrigerant mass needs to circulate** to deliver the same cooling duty — one of the practical reasons isobutane systems use such small charge quantities (typically 40–90g) compared to older HFC systems.

In [ ]:
# FINAL CELL — Push this notebook to GitHub
from github import Github, Auth
from google.colab import userdata, _message
import json

token = userdata.get('GITHUB_TOKEN')
auth = Auth.Token(token)
g = Github(auth=auth)
repo = g.get_repo("MaheshaGonal/refrigeration-simulation-python")

nb_data = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
content = json.dumps(nb_data['ipynb'], indent=1)

filename = "day01_first_property.ipynb"

try:
    existing = repo.get_contents(filename)
    repo.update_file(filename, "Add Day 01 - first property extraction (R134a vs R600a)", content, existing.sha)
    print("Updated existing file on GitHub")
except Exception:
    repo.create_file(filename, "Add Day 01 - first property extraction (R134a vs R600a)", content)
    print("Created new file on GitHub")

print("GitHub repo: https://github.com/MaheshaGonal/refrigeration-simulation-python")
